In [1]:
import sys
sys.path.append(r"C:\Users\zhaoh\Desktop\CDS")
import os 
import torch
import numpy as np
import pandas as pd
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from torchvision import transforms
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from src.utils import load_video_data, evaluate, extract_faces_preserve_structure_mtcnn
from src.data.image.data_loader import ImageDataset
from sklearn.utils.class_weight import compute_class_weight
import torchvision.models as models
from src.models.image.resnet import ResNet50

c:\Users\zhaoh\Desktop\CDS\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import re
from unidecode import unidecode

def clean_utterance(text):
    text = str(text)
    # Normalize smart quotes and apostrophes
    text = text.replace('‘','\'').replace('’','\'').replace('','\'').replace('','\'')
    text = text.replace('“','"').replace('”','"').replace('','"').replace('','"')
    
    # Replace ellipsis and special dots
    text = text.replace('…','...').replace('','...')
    
    # Replace dashes
    text = text.replace('—','-').replace('–','-').replace('','-').replace('','-')
    
    # Replace non-breaking space with normal space
    text = text.replace('\xa0',' ').replace(' ',' ')
    
    # Optional: normalize accented characters
    text = unidecode(text)
    
    # Remove any other remaining non-ASCII characters
    text = re.sub(r'[^\x00-\x7F]+', '', text)
    
    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

In [3]:
# Already loaded CSV inside load_video_data
test_video_dirs, test_labels, test_emotion_to_idx = load_video_data(
    metadata_csv="C:/Users/zhaoh/Desktop/processed_test/metadata.csv",
    frames_root_dir="C:/Users/zhaoh/Desktop/processed_test/frames_test/"
)


In [4]:
import os
import torch
import numpy as np
from torch.utils.data import DataLoader
from torchvision import transforms
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

# =====================
# Device
# =====================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# =====================
# Paths
# =====================
test_metadata_csv = "C:/Users/zhaoh/Desktop/processed_test/metadata.csv"
checkpoint_path = "C:/Users/zhaoh/Desktop/CDS/text_checkpoint/checkpoint-3750"
image_weights_path = "resnet_model_weights_extracted_5.pth"
frames_root_dir = "C:/Users/zhaoh/Desktop/processed_test/frames_test/"

# =====================
# Load test metadata
# =====================
import pandas as pd

test_df = pd.read_csv(test_metadata_csv)[["video_id","utterance", "emotion"]]
label_list = sorted(test_df["emotion"].unique())
label2id = {l: i for i, l in enumerate(label_list)}
id2label = {i: l for l, i in label2id.items()}
test_df["label"] = test_df["emotion"].map(label2id)

# Build video_id -> clean_text mapping
def clean_utterance(text):
    import re
    from unidecode import unidecode
    text = str(text)
    text = text.replace('…', '...').replace('\xa0', ' ')
    text = unidecode(text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

video_to_text = {row["video_id"]: clean_utterance(row["utterance"])
                 for _, row in test_df.iterrows()}

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# =====================
# Text tokenizer + dataset
# =====================
from datasets import Dataset

test_dataset = Dataset.from_pandas(test_df)
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("bhadresh-savani/bert-base-uncased-emotion")

def tokenize_function(example):
    return tokenizer(example["utterance"], truncation=True, padding="max_length", max_length=128)

test_dataset = test_dataset.map(tokenize_function, batched=True)
test_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

# =====================
# Load text model
# =====================
from transformers import AutoModelForSequenceClassification
text_model = AutoModelForSequenceClassification.from_pretrained(checkpoint_path)
text_model = text_model.to(device)
text_model.eval()

# =====================
# Load image model
# =====================
# Assume you have defined ResNet50 class elsewhere
model_resnet = ResNet50(num_classes=len(label_list))
model_resnet.load_state_dict(torch.load(image_weights_path, map_location=device))
model_resnet = model_resnet.to(device)
model_resnet.eval()

# =====================
# Prepare image dataset
# =====================
from torch.utils.data import Dataset as TorchDataset

test = ImageDataset(
    video_dirs=test_video_dirs,
    labels=test_labels,
    transform=transform,
    n_frames=8
)

image_loader = DataLoader(
    test,
    batch_size=8,  # number of videos per batch
    shuffle=False,
    num_workers=0,
)

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

texts_aligned = []
aligned_labels = []
aligned_video_dirs = []

for video_dir, label in zip(test_video_dirs, test_labels):
    video_id = os.path.basename(video_dir)
    
    # Make sure video_id exists in metadata
    if video_id not in video_to_text:
        continue
    
    # Make sure frames folder exists and is not empty
    if not os.path.exists(video_dir) or len(os.listdir(video_dir)) == 0:
        continue
    
    # All checks passed — add to aligned lists
    texts_aligned.append(video_to_text[video_id])
    aligned_labels.append(label)
    aligned_video_dirs.append(video_dir)

# =====================
# Helper functions to get probabilities
# =====================
def get_text_probs(model, tokenizer, texts, device):
    model.eval()
    probs_list = []
    with torch.no_grad():
        for text in texts:
            inputs = tokenizer(text, return_tensors="pt", truncation=True, padding="max_length", max_length=128)
            inputs = {k: v.to(device) for k, v in inputs.items()}
            outputs = model(**inputs)
            probs = torch.softmax(outputs.logits, dim=1)
            probs_list.append(probs.cpu())
    return torch.cat(probs_list, dim=0)

def get_image_probs(model, loader, device):
    model.eval()
    probs_list = []
    with torch.no_grad():
        for imgs, _ in loader:
            imgs = imgs.to(device)
            outputs = model(imgs)
            probs = torch.softmax(outputs, dim=1)
            probs_list.append(probs.cpu())
    return torch.cat(probs_list, dim=0)

# =====================
# Get probabilities
# =====================
text_probs = get_text_probs(text_model, tokenizer, texts_aligned, device)
image_probs = get_image_probs(model_resnet, image_loader, device)

# =====================
# Fuse probabilities
# =====================
text_weight = 0.6
image_weight = 0.4  # change >0 to include image info

fused_probs = text_weight * text_probs + image_weight * image_probs
predictions = torch.argmax(fused_probs, dim=1).numpy()
true_labels = np.array(aligned_labels)

# =====================
# Evaluate
# =====================
print("Accuracy:", accuracy_score(true_labels, predictions))
print("Weighted F1:", f1_score(true_labels, predictions, average="weighted"))
print("\nClassification Report:\n", classification_report(true_labels, predictions))
print("\nConfusion Matrix:\n", confusion_matrix(true_labels, predictions))

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 1026.18it/s, Materializing param=classifier.weight]                                     


Accuracy: 0.5907643312101911
Weighted F1: 0.5939169708104091

Classification Report:
               precision    recall  f1-score   support

           0       0.42      0.37      0.39       163
           1       0.37      0.29      0.32        35
           2       0.17      0.16      0.16        25
           3       0.50      0.59      0.54       197
           4       0.78      0.71      0.74       618
           5       0.34      0.37      0.35        89
           6       0.48      0.62      0.54       129

    accuracy                           0.59      1256
   macro avg       0.44      0.44      0.44      1256
weighted avg       0.60      0.59      0.59      1256


Confusion Matrix:
 [[ 60   4   7  20  31  16  25]
 [  6  10   0   4   8   3   4]
 [  5   1   4   1   8   4   2]
 [ 18   3   1 116  35   9  15]
 [ 35   4   7  64 439  32  37]
 [  9   4   3  11  25  33   4]
 [ 11   1   2  17  17   1  80]]
